# Portfolio Manager Debug Notebook

Clean, executable pipeline for mutual fund portfolio construction with interactive debugging.

## 1. Environment Setup

In [21]:
%load_ext autoreload
%autoreload 2

import logging
import numpy as np
import pandas as pd
from pathlib import Path

np.random.seed(42)
pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 220)

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## 2. Paths and Parameters

Edit these to change behavior without modifying logic.

In [22]:
# Paths
RAW_FUNDS_PATH = Path('../mutualfunds/raw_funds.tsv')
FUND_INFO_DIR = Path('../mutualfunds/fund_info')
OUTPUT_DIR = Path('tuning_results/portfolio_debug')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Portfolio parameters
NUM_FUNDS = 5
RISK_PROFILE = 'aggresive'  # conservative | moderate | aggressive
FUND_NAMES = [
    'HDFC Mid Cap Dir Gr',
    'Edelweiss Mid Cap Dir Gr',
    'Nippon India Growth Mid Cap Dir Gr',
    'Invesco India Mid Cap Dir Gr',
    'ICICI Pru MidCap Dir Gr'
]

# Constraints
MAX_OVERLAP_PCT = 40.0

logger.info(f'Loaded parameters: {NUM_FUNDS} funds, {RISK_PROFILE} profile')

INFO:__main__:Loaded parameters: 5 funds, aggresive profile


## 3. In-Notebook Portfolio Logic

In [31]:
import pandas as pd

def safe_float(x):
    try:
        return float(x)
    except:
        return None


def quality_filter(fund_isins, fund_info_dir):
    """
    Improved quality filter:
    - Sharpe_3Y >= category average
    - StdDev_3Y <= 1.2 * category average
    - Returns_3Y > 0
    - Sortino_3Y >= category average (NEW)
    - Skip missing data safely
    """
    filtered = []
    
    for isin in fund_isins:
        risk_file = fund_info_dir / f"risk_metrics_{isin}.tsv"
        
        if not risk_file.exists():
            continue
        
        risk_df = pd.read_csv(risk_file, sep='\t')
        if risk_df.empty:
            continue
        
        row = risk_df.iloc[0]

        sharpe_3y = safe_float(row.get('sharpe_3y'))
        std_3y = safe_float(row.get('std_3y'))
        returns_3y = safe_float(row.get('returns_3y'))
        sharpe_cat_avg_3y = safe_float(row.get('sharpe_cat_avg_3y'))
        std_cat_avg_3y = safe_float(row.get('std_cat_avg_3y'))

        sortino_3y = safe_float(row.get('sortino_3y'))
        sortino_cat_avg_3y = safe_float(row.get('sortino_cat_avg_3y'))

        # 🚨 Skip incomplete data
        if None in [sharpe_3y, std_3y, returns_3y, sharpe_cat_avg_3y, std_cat_avg_3y]:
            continue

        if (
            sharpe_3y >= sharpe_cat_avg_3y and
            std_3y <= std_cat_avg_3y * 1.2 and
            returns_3y > 0 and
            (sortino_3y is None or sortino_cat_avg_3y is None or sortino_3y >= sortino_cat_avg_3y)
        ):
            filtered.append(isin)
    
    return filtered

In [32]:
import pandas as pd

def rank_funds(filtered_funds, fund_info_dir):
    rows = []

    for isin in filtered_funds:
        risk_file = fund_info_dir / f"risk_metrics_{isin}.tsv"
        if not risk_file.exists():
            continue

        risk_df = pd.read_csv(risk_file, sep='\t')
        if risk_df.empty:
            continue

        row = risk_df.iloc[0]

        def safe_float(x):
            try:
                return float(x)
            except:
                return None

        sharpe = safe_float(row.get('sharpe_3y'))
        sharpe_avg = safe_float(row.get('sharpe_cat_avg_3y'))

        sortino = safe_float(row.get('sortino_3y'))
        sortino_avg = safe_float(row.get('sortino_cat_avg_3y'))

        returns = safe_float(row.get('returns_3y'))
        returns_avg = safe_float(row.get('returns_cat_avg_3y'))

        std = safe_float(row.get('std_3y'))
        std_avg = safe_float(row.get('std_cat_avg_3y'))

        if None in [sharpe, sharpe_avg, returns, returns_avg, std, std_avg]:
            continue

        sharpe_score = sharpe / sharpe_avg if sharpe_avg else 0
        sortino_score = sortino / sortino_avg if sortino_avg else 0
        return_score = returns / returns_avg if returns_avg else 0
        std_score = std / std_avg if std_avg else 1

        score = (
            0.4 * sharpe_score +
            0.2 * sortino_score +
            0.2 * return_score -
            0.2 * std_score
        )

        rows.append({
            "ISIN": isin,
            "score": score,
            "sharpe_3y": sharpe,
            "sortino_3y": sortino,
            "returns_3y": returns,
            "std_3y": std,
            "sharpe_score": sharpe_score,
            "sortino_score": sortino_score,
            "return_score": return_score,
            "std_score": std_score
        })

    df = pd.DataFrame(rows)

    if not df.empty:
        df = df.sort_values(by="score", ascending=False).reset_index(drop=True)

    return df

In [49]:
import pandas as pd

def safe_float(x):
    try:
        return float(str(x).replace(',', ''))
    except:
        return 0.0


def compute_overlap_matrix(ranked_df, fund_info_dir):
    """
    Compute overlap matrix using ranked_df (from rank_funds)
    """

    isins = ranked_df["ISIN"].tolist()
    holdings = {}
    missing = []

    # 🔹 Load holdings
    for isin in isins:
        holdings_file = fund_info_dir / f"holdings_{isin}.tsv"

        if not holdings_file.exists():
            holdings[isin] = {}
            missing.append(isin)
            continue

        df = pd.read_csv(holdings_file, sep='\t')

        if df.empty:
            holdings[isin] = {}
            missing.append(isin)
            continue

        df['weight'] = df['weight'].apply(safe_float)

        df_agg = df.groupby('stock_name')['weight'].sum().reset_index()

        holdings[isin] = df_agg.set_index('stock_name')['weight'].to_dict()

    # 🔹 Initialize matrix
    n = len(isins)
    overlap = pd.DataFrame(0.0, index=isins, columns=isins)

    # 🔹 Compute overlap
    for i in range(n):
        for j in range(i, n):
            fund_a = isins[i]
            fund_b = isins[j]

            if i == j:
                overlap.loc[fund_a, fund_b] = 100.0
                continue

            overlap_ab = 0.0

            stocks_a = holdings[fund_a]
            stocks_b = holdings[fund_b]

            all_stocks = set(stocks_a.keys()) | set(stocks_b.keys())

            for stock in all_stocks:
                w_a = stocks_a.get(stock, 0.0)
                w_b = stocks_b.get(stock, 0.0)
                overlap_ab += min(w_a, w_b)

            # 🔥 NORMALIZATION STEP (HERE)
            total_a = sum(stocks_a.values())
            total_b = sum(stocks_b.values())
            
            if min(total_a, total_b) > 0:
                overlap_ab = overlap_ab / min(total_a, total_b)
            else:
                overlap_ab = 0.0
            overlap_ab *= 100
            
            overlap.loc[fund_a, fund_b] = overlap_ab
            overlap.loc[fund_b, fund_a] = overlap_ab

    # 🔹 Debug missing
    if missing:
        print(f"⚠️ Missing holdings for {len(missing)} funds")

    return overlap

In [50]:
import pandas as pd

def safe_float(x):
    try:
        return float(x)
    except:
        return None


def optimize_portfolio(ranked_df, overlap_matrix, risk_profile, fund_info_dir):
    """
    Portfolio optimizer:
    - Uses ranked_df (DataFrame)
    - Removes high overlap (>0.4)
    - Allocates via Sharpe/Std
    """

    isins = ranked_df["ISIN"].tolist()

    # 🔹 Step 1: Number of funds
    if risk_profile == 'conservative':
        num_select = min(3, len(isins))
    elif risk_profile == 'moderate':
        num_select = min(4, len(isins))
    else:
        num_select = min(5, len(isins))

    selected = []

    # 🔹 Step 2: Greedy selection (better than remove-after)
    for isin in isins:
        if len(selected) >= num_select:
            break

        add = True
        for s in selected:
            if overlap_matrix.loc[isin, s] > 40:
                add = False
                break

        if add:
            selected.append(isin)

    # 🔹 Fallback (ensure minimum funds)
    if len(selected) < 2:
        selected = isins[:num_select]

    # 🔹 Step 3: Weight allocation
    weights = {}
    total_ratio = 0.0

    for isin in selected:
        risk_file = fund_info_dir / f"risk_metrics_{isin}.tsv"

        if not risk_file.exists():
            continue

        risk_df = pd.read_csv(risk_file, sep='\t')
        if risk_df.empty:
            continue

        row = risk_df.iloc[0]

        sharpe = safe_float(row.get('sharpe_3y'))
        std = safe_float(row.get('std_3y'))

        if sharpe is None or std is None or std == 0:
            continue

        ratio = sharpe / std
        weights[isin] = ratio
        total_ratio += ratio

    # 🔹 Normalize weights
    if total_ratio > 0:
        weights = {k: v / total_ratio for k, v in weights.items()}
    else:
        n = len(selected)
        if n > 0:
            equal_weight = 1.0 / n
            weights = {isin: equal_weight for isin in selected}

    return weights

In [51]:
logger.info('Portfolio manager logic loaded in-notebook (no external imports)')

INFO:__main__:Portfolio manager logic loaded in-notebook (no external imports)


## 4. Load and Validate Data

In [52]:
# Load raw fund data
raw_df = pd.read_csv(RAW_FUNDS_PATH, sep='\t')
raw_df.columns = [c.strip() for c in raw_df.columns]

assert 'schemeName' in raw_df.columns, 'Missing schemeName column'
assert 'isin' in raw_df.columns, 'Missing isin column'

print(f'Loaded {len(raw_df)} fund records')
print(f'Columns: {list(raw_df.columns)}')

Loaded 924 fund records
Columns: ['annualized3Y', 'fundCode', 'invCategory', 'invType', 'isin', 'ltd', 'mc30', 'rank', 'rating', 'risk', 'schemeCode', 'schemeName', 'schemePlan', 'sipReturns', 'slug_url', 'trailingReturns', 'updatedDate', 'yearlyReturns']


## 5. Resolve Fund Names to ISINs

In [53]:
def resolve_fund_names(names, raw_df):
    """Resolve fund names to ISINs."""
    rows = []
    for name in names:
        match = raw_df[raw_df['schemeName'].str.lower() == name.lower()]
        isin = str(match.iloc[0]['isin']) if not match.empty else None
        rows.append({'fund_name': name, 'isin': isin, 'resolved': isin is not None})
    return pd.DataFrame(rows)

resolve_df = resolve_fund_names(FUND_NAMES[:NUM_FUNDS], raw_df)
print('Fund Resolution:')
print(resolve_df)

fund_isins = resolve_df.loc[resolve_df['resolved'], 'isin'].tolist()
assert fund_isins, 'No funds resolved. Check fund names.'
print(f'\nValid ISINs: {fund_isins}')

Fund Resolution:
                            fund_name          isin  resolved
0                 HDFC Mid Cap Dir Gr  INF179K01XQ0      True
1            Edelweiss Mid Cap Dir Gr  INF843K01AO4      True
2  Nippon India Growth Mid Cap Dir Gr  INF204K01E54      True
3        Invesco India Mid Cap Dir Gr  INF205K01MV6      True
4             ICICI Pru MidCap Dir Gr  INF109K011N7      True

Valid ISINs: ['INF179K01XQ0', 'INF843K01AO4', 'INF204K01E54', 'INF205K01MV6', 'INF109K011N7']


## 6. Quality Filter → Ranking → Overlap → Optimization

In [54]:
# Step 1: Quality Filter
filtered_isins = quality_filter(fund_isins, FUND_INFO_DIR)
print(f'Step 1 - Quality Filter:')
print(f'  Input: {len(fund_isins)}, Passed: {len(filtered_isins)}')

assert filtered_isins, 'No funds passed quality filter'


# Step 2: Ranking → RETURNS DATAFRAME
ranked_df = rank_funds(filtered_isins, FUND_INFO_DIR)
print(f'\nStep 2 - Ranking:')

if ranked_df.empty:
    raise ValueError("Ranking returned empty DataFrame")

print(ranked_df.head())


# Step 3: Overlap Matrix → TAKES DATAFRAME
overlap_df = compute_overlap_matrix(ranked_df, FUND_INFO_DIR)
print(f'\nStep 3 - Overlap Matrix:')
print(overlap_df.round(2))   # cleaner display


# Step 4: Portfolio Optimization → TAKES DATAFRAME
portfolio = optimize_portfolio(ranked_df, overlap_df, RISK_PROFILE, FUND_INFO_DIR)

print(f'\nStep 4 - Portfolio Optimization:')
for isin, weight in portfolio.items():
    print(f'  {isin}: {weight:.2%}')

Step 1 - Quality Filter:
  Input: 5, Passed: 5

Step 2 - Ranking:
           ISIN     score  sharpe_3y  sortino_3y  returns_3y  std_3y  sharpe_score  sortino_score  return_score  std_score
0  INF179K01XQ0  0.905022       1.39        2.44       22.60   13.69      1.376238       1.478788      1.169167   0.875320
1  INF204K01E54  0.832374       1.29        2.28       23.32   15.34      1.277228       1.381818      1.206415   0.980818
2  INF843K01AO4  0.825558       1.30        2.18       23.45   15.34      1.287129       1.321212      1.213140   0.980818
3  INF109K011N7  0.786765       1.23        2.15       23.30   15.80      1.217822       1.303030      1.205380   1.010230
4  INF205K01MV6  0.778347       1.25        2.04       23.85   16.48      1.237624       1.236364      1.233833   1.053708

Step 3 - Overlap Matrix:
              INF179K01XQ0  INF204K01E54  INF843K01AO4  INF109K011N7  INF205K01MV6
INF179K01XQ0        100.00         32.33         30.66         13.68         21.63
INF2

## 8. Results Summary and Diagnostics

In [48]:
# Add fund names to portfolio
mapping_df = raw_df[['schemeName', 'isin']].drop_duplicates()
portfolio_named = portfolio_df.merge(mapping_df, on='isin', how='left')

print('Final Portfolio Assignment:')
print(portfolio_named[['schemeName', 'isin', 'weight']].to_string())

# Diagnostics
print(f'\nDiagnostics:')
print(f'  Total funds in portfolio: {len(portfolio_df)}')
print(f'  Risk profile applied: {RISK_PROFILE}')
print(f'  Max overlap constraint: {MAX_OVERLAP_PCT}%')

# Overlap diagnostics
if not overlap_df.empty and len(overlap_df) > 1:
    print(f'\nPairwise Overlaps:')
    for i in range(len(overlap_df.index)):
        for j in range(i + 1, len(overlap_df.columns)):
            a, b = overlap_df.index[i], overlap_df.columns[j]
            ov = float(overlap_df.loc[a, b])
            print(f'  {a} ↔ {b}: {ov:.1f}%')

Final Portfolio Assignment:
                           schemeName          isin    weight
0                 HDFC Mid Cap Dir Gr  INF179K01XQ0  0.239427
1            Edelweiss Mid Cap Dir Gr  INF843K01AO4  0.199839
2  Nippon India Growth Mid Cap Dir Gr  INF204K01E54  0.198301
3             ICICI Pru MidCap Dir Gr  INF109K011N7  0.183573
4        Invesco India Mid Cap Dir Gr  INF205K01MV6  0.178860

Diagnostics:
  Total funds in portfolio: 5
  Risk profile applied: aggresive
  Max overlap constraint: 40.0%

Pairwise Overlaps:
  INF179K01XQ0 ↔ INF204K01E54: 32.3%
  INF179K01XQ0 ↔ INF843K01AO4: 30.7%
  INF179K01XQ0 ↔ INF109K011N7: 13.7%
  INF179K01XQ0 ↔ INF205K01MV6: 21.6%
  INF204K01E54 ↔ INF843K01AO4: 45.1%
  INF204K01E54 ↔ INF109K011N7: 21.6%
  INF204K01E54 ↔ INF205K01MV6: 25.9%
  INF843K01AO4 ↔ INF109K011N7: 26.7%
  INF843K01AO4 ↔ INF205K01MV6: 19.4%
  INF109K011N7 ↔ INF205K01MV6: 10.5%


## 9. Export Results

In [ ]:
# Export intermediate results
resolve_df.to_csv(OUTPUT_DIR / 'resolve_df.csv', index=False)
portfolio_named.to_csv(OUTPUT_DIR / 'portfolio.csv', index=False)
overlap_df.to_csv(OUTPUT_DIR / 'overlap_matrix.csv')

# Export run summary
summary = {
    'run_date': pd.Timestamp.now(),
    'num_funds_requested': NUM_FUNDS,
    'risk_profile': RISK_PROFILE,
    'funds_resolved': int(resolve_df['resolved'].sum()),
    'funds_after_quality_filter': len(filtered_isins),
    'funds_in_final_portfolio': len(portfolio_df),
    'max_overlap_constraint': MAX_OVERLAP_PCT,
}

summary_df = pd.DataFrame([summary])
summary_df.to_csv(OUTPUT_DIR / 'run_summary.csv', index=False)

print(f'Results exported to {OUTPUT_DIR}')
print(f'  - resolve_df.csv')
print(f'  - portfolio.csv')
print(f'  - overlap_matrix.csv')
print(f'  - run_summary.csv')